# 22b — Attacks on the final system (and an exploratory abstention rule)

**Goal.** Notebook 13 showed that typos and paraphrases barely hurt the *old single-model* detector at its *old 1% logit threshold*. The final system is different: a three-seed ensemble judged on the p scale with the strict flag line (p ≤ 0.00235). Phase 2 showed that this line sits close to the score ceiling, and that 5% typos changed the official verdict on 13.5% of a *unanimous-AI* sample. Here the attacks are re-run on the final system, on **random** samples (not chosen by jury state).

**Declared before computing** (`docs/jury_22b_declaration.md`, commit `c4c8560`): samples, attacks, statistics E1–E5, and the abstention rule for the exploratory part. Held-out groups are descriptive. Nothing here changes the final system.

**The attacks** (all keep the meaning of the essay): typos in 2%, 5% or 10% of words; a pretrained T5 paraphraser at temperature 1.0 (light, ~15% of words changed) or 1.5 (heavy, ~43%); and T5 (T=1.0) followed by 5% typos. Same seeds as notebook 13.

**Vocabulary.** *Flagged*: the official ensemble's p ≤ 0.00235. *Above usual*: 0.00235 < p ≤ 0.01032. *Verdict flip* (E4): an essay flagged before the attack that is not flagged after (for students: not flagged before, flagged after).

In [1]:
import os, sys, gc, time
os.environ['HF_HUB_OFFLINE'] = '1'; os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
sys.path.append('..'); sys.path.append('../scripts')
import numpy as np, pandas as pd, torch, transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from src.data import load_splits
from src.conformal import tail_p
from src.ensemble import PEnsemble
from src.evaluate import wilson_interval
from src.paraphraser import T5Paraphraser, inject_typos, words_changed
from jury_export import FP16, logits as juror_logits
transformers.logging.set_verbosity_error()
SEED = 42; R = '../results/'
TAU_FLAG, TAU_ABOVE = 0.00235, 0.01032
SEEDS = list(range(42, 62)); COLS = [f's{s}' for s in SEEDS]
pd.set_option('display.width', 230)

## Step 1 — The essays under attack

Same recipe as notebook 13 (`random_state=42`, on-topic only), plus one new group so unseen-topic students are covered:
- all on-topic AI essays in `test` (473);
- 300 on-topic AI essays from the held-out generators (descriptive);
- 300 on-topic `test` students (the same sample as notebook 13);
- 300 on-topic students from the held-out prompts (new; descriptive).

In [2]:
data = load_splits(); on = data[data['on_topic']]
ai_test = on[(on.split == 'test') & (on.label == 1)]
ai_gen = on[on.split == 'heldout_generator'].sample(300, random_state=SEED)
h_test = on[(on.split == 'test') & (on.label == 0)].sample(300, random_state=SEED)
h_hp = on[(on.split == 'heldout_prompt') & (on.label == 0)].sample(300, random_state=SEED)
essays = pd.concat([ai_test.assign(group='AI, test'), ai_gen.assign(group='AI, held-out generators'),
                    h_test.assign(group='Students, test'), h_hp.assign(group='Students, held-out prompts')])
GROUPS = ['AI, test', 'AI, held-out generators', 'Students, test', 'Students, held-out prompts']
assert essays.index.is_unique
print(essays.group.value_counts().reindex(GROUPS).to_dict())

{'AI, test': 473, 'AI, held-out generators': 300, 'Students, test': 300, 'Students, held-out prompts': 300}


## Step 2 — Build the attacked copies

Each attack makes a full rewritten copy of every essay (T5 works sentence by sentence; its spacing artefacts are cleaned, as in notebook 13). The texts are not saved: only the scores are.

In [3]:
texts = essays['text'].tolist(); ids = essays.index.to_numpy()
versions = {'original': texts}
for rate in (0.02, 0.05, 0.10):
    versions[f'typos {rate:.0%}'] = [inject_typos(t, rate, seed=SEED + i) for i, t in enumerate(texts)]
t0 = time.time(); t5 = T5Paraphraser()
versions['T5 paraphrase (T=1.0)'] = t5.paraphrase_essays(texts, temperature=1.0, seed=SEED)
versions['T5 paraphrase (T=1.5)'] = t5.paraphrase_essays(texts, temperature=1.5, seed=SEED)
del t5; gc.collect(); torch.cuda.empty_cache()
versions['T5 (T=1.0) + typos 5%'] = [inject_typos(t, 0.05, seed=SEED + i) for i, t in enumerate(versions['T5 paraphrase (T=1.0)'])]
print(f'{len(versions)} versions of {len(texts)} essays in {(time.time() - t0) / 60:.1f} min')
words = {v: np.array([words_changed(a, b) for a, b in zip(texts, versions[v])]) for v in versions}
print({v: round(100 * words[v][essays.label.to_numpy() == 1].mean(), 1) for v in versions}, '(% words changed, AI essays)')

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

7 versions of 1373 essays in 30.0 min


{'original': np.float64(0.0), 'typos 2%': np.float64(1.6), 'typos 5%': np.float64(4.0), 'typos 10%': np.float64(8.2), 'T5 paraphrase (T=1.0)': np.float64(15.3), 'T5 paraphrase (T=1.5)': np.float64(42.6), 'T5 (T=1.0) + typos 5%': np.float64(19.0)} (% words changed, AI essays)


## Step 3 — Score every version with all 20 jurors

The 20 fp16 serving copies of Phase 1 score every version (one juror at a time on the GPU). The official ensemble is built from jurors 42–44; the jury's flag count uses all 20. Only the logits are saved.

In [4]:
tok = AutoTokenizer.from_pretrained(FP16 / 's42'); pad = tok.pad_token_id
enc = {v: tok(vs, truncation=True, max_length=512)['input_ids'] for v, vs in versions.items()}
out = {v: {} for v in versions}; t0 = time.time()
for s in SEEDS:
    m = AutoModelForSequenceClassification.from_pretrained(FP16 / f's{s}', torch_dtype=torch.float16).cuda().eval()
    for v in versions: out[v][f's{s}'] = juror_logits(m, enc[v], pad, autocast=False)
    del m; torch.cuda.empty_cache()
print(f'scored {len(versions)} x {len(texts)} essays with 20 jurors in {(time.time() - t0) / 60:.1f} min')
rows = []
for v in versions:
    d = pd.DataFrame(out[v]); d.insert(0, 'row_id', ids); d.insert(0, 'attack', v); d.insert(2, 'group', essays.group.to_numpy()); rows.append(d)
LG = pd.concat(rows, ignore_index=True)
LG.to_csv(R + '22b_attack_juror_logits.csv', index=False)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

... (11 outputs left out)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

scored 7 x 1373 essays with 20 jurors in 12.5 min


## Step 4 — Turn logits into verdicts

Each juror's logit becomes its own tail probability against its own validation humans (the same calibration as Phase 2); the ensemble is `PEnsemble` on jurors 42–44. A check on the originals: the verdicts of the unattacked essays must equal the verdicts stored by Phase 2.

In [5]:
vt = pd.read_csv(R + '22_juror_scores.csv').set_index('row_id')
vh = vt[(vt.split == 'val') & (vt.label == 0)][COLS]
V = {c: vh[c].to_numpy() for c in COLS}
ens = PEnsemble(np.column_stack([V[c] for c in COLS[:3]]))
def verdicts(df):
    p = ens.p_ens(df[COLS[:3]].to_numpy())
    nflag = sum((tail_p(df[c].to_numpy(), V[c]) <= TAU_FLAG).astype(int) for c in COLS)
    return pd.DataFrame({'p_ens': p, 'flagged': p <= TAU_FLAG, 'flag_or_above': p <= TAU_ABOVE, 'n_flag': nflag}, index=df.index)
LG = LG.reset_index(drop=True); VD = pd.concat([LG[['attack', 'row_id', 'group']], verdicts(LG)], axis=1)
pe = pd.read_csv(R + '22_jury_per_essay.csv').set_index('row_id')
o = VD[VD.attack == 'original'].set_index('row_id')
assert (o.flagged.to_numpy() == pe.loc[o.index, 'ensemble_flagged'].to_numpy()).all() and (o.n_flag.to_numpy() == pe.loc[o.index, 'n_flag_20'].to_numpy()).all()
print('unattacked verdicts and jury counts match Phase 2 for all', len(o), 'essays')

unattacked verdicts and jury counts match Phase 2 for all 1373 essays


## Step 5 — E1–E4: the final system under attack

For every group and attack: % flagged, % flagged-or-above-usual (Wilson 95% intervals), the jury's mean flag count, and the paired verdict flip.

In [6]:
def wil(k, n):
    lo, hi = wilson_interval(int(k), int(n)); return 100 * k / n, 100 * lo, 100 * hi
res = []
base = VD[VD.attack == 'original'].set_index('row_id')
for g in GROUPS:
    is_ai = g.startswith('AI')
    b = base[base.group == g]
    for a in versions:
        d = VD[(VD.attack == a) & (VD.group == g)].set_index('row_id').loc[b.index]
        n = len(d); r = dict(group=g, attack=a, n=n)
        for name, col in (('flagged', 'flagged'), ('flag_or_above', 'flag_or_above')):
            k = int(d[col].sum()); r[f'{name}_k'] = k; r[f'{name}_pct'], r[f'{name}_lo'], r[f'{name}_hi'] = wil(k, n)
        r['mean_jurors_flagging_before'] = b.n_flag.mean(); r['mean_jurors_flagging_after'] = d.n_flag.mean()
        if is_ai:                                                   # flip = flagged before, not flagged after
            denom = b.flagged.to_numpy(); flip = denom & ~d.flagged.to_numpy()
        else:                                                       # flip = not flagged before, flagged after
            denom = ~b.flagged.to_numpy(); flip = denom & d.flagged.to_numpy()
        r['flip_k'] = int(flip.sum()); r['flip_denominator'] = int(denom.sum())
        r['flip_pct'], r['flip_lo'], r['flip_hi'] = wil(flip.sum(), max(denom.sum(), 1)) if denom.sum() else (np.nan,) * 3
        r['words_changed_pct'] = 100 * words[a][(essays.group == g).to_numpy()].mean()
        res.append(r)
RES = pd.DataFrame(res); RES.to_csv(R + '22b_attack_final.csv', index=False)
def fmt(r, c): return f"{r[c + '_pct']:.1f} [{r[c + '_lo']:.1f}, {r[c + '_hi']:.1f}]"
for g in GROUPS:
    t = RES[RES.group == g]
    print('\n' + g, f'(n={t.n.iloc[0]})')
    print(pd.DataFrame({'flagged %': t.apply(lambda r: fmt(r, 'flagged'), axis=1), 'flagged or above usual %': t.apply(lambda r: fmt(r, 'flag_or_above'), axis=1),
                        'jurors flagging before -> after': t.apply(lambda r: f"{r.mean_jurors_flagging_before:.1f} -> {r.mean_jurors_flagging_after:.1f}", axis=1),
                        'verdict flip %': t.apply(lambda r: fmt(r, 'flip') + f" ({r.flip_k}/{r.flip_denominator})", axis=1),
                        'words changed %': t.words_changed_pct.round(1)}).set_index(t.attack).to_string())


AI, test (n=473)
                               flagged % flagged or above usual % jurors flagging before -> after              verdict flip %  words changed %
attack                                                                                                                                        
original               99.2 [97.8, 99.7]        99.6 [98.5, 99.9]                    19.8 -> 19.8      0.0 [0.0, 0.8] (0/469)              0.0
typos 2%               97.9 [96.2, 98.8]        99.4 [98.2, 99.8]                    19.8 -> 19.3      1.3 [0.6, 2.8] (6/469)              1.6
typos 5%               96.6 [94.6, 97.9]        99.4 [98.2, 99.8]                    19.8 -> 18.0     2.6 [1.5, 4.4] (12/469)              4.0
typos 10%              90.1 [87.0, 92.4]        98.7 [97.3, 99.4]                    19.8 -> 16.0    9.2 [6.9, 12.1] (43/469)              8.0
T5 paraphrase (T=1.0)  98.9 [97.5, 99.5]        99.6 [98.5, 99.9]                    19.8 -> 19.7      0.2 [0.0, 1.2] (1/469

## Step 6 — Next to notebook 13 (E5)

Notebook 13 scored a *single* DistilRoBERTa at its *old 1% logit threshold* (logit 4.67). Its figures apply to that threshold only. The T5 texts here were regenerated together with a new group, so this is a side-by-side, not a paired comparison.

In [7]:
old = pd.read_csv(R + '13_attack.csv').set_index('attack')
oc = {'AI, test': 'DistilRoBERTa: % AI caught (test)', 'AI, held-out generators': 'DistilRoBERTa: % AI caught (held-out gen)', 'Students, test': 'DistilRoBERTa: % humans flagged'}
cmp = []
for a in versions:
    r = {'attack': a}
    for g, c in oc.items():
        n_ = RES[(RES.group == g) & (RES.attack == a)].iloc[0]
        r[g + ': old single model (13)'] = round(old.loc[a, c], 1); r[g + ': final ensemble'] = round(n_.flagged_pct, 1)
    cmp.append(r)
CMP = pd.DataFrame(cmp).set_index('attack'); CMP.to_csv(R + '22b_vs_13.csv'); print(CMP.T.to_string())

attack                                          original  typos 2%  typos 5%  typos 10%  T5 paraphrase (T=1.0)  T5 paraphrase (T=1.5)  T5 (T=1.0) + typos 5%
AI, test: old single model (13)                     99.6      99.4      99.4       98.9                   99.6                   96.4                   98.9
AI, test: final ensemble                            99.2      97.9      96.6       90.1                   98.9                   81.4                   87.7
AI, held-out generators: old single model (13)     100.0     100.0      99.3       98.3                   99.7                   96.0                   98.7
AI, held-out generators: final ensemble             94.7      83.7      67.3       43.7                   93.3                   52.7                   61.7
Students, test: old single model (13)                0.7       0.3       0.0        0.0                    2.7                    0.3                    0.3
Students, test: final ensemble                       0.0  

## Step 7 — Claims audit rows

The old audit file (`15_claims_audit.csv`) is not edited. `22b_claims_audit.csv` lists notebook 13's robustness claims with their scope, and the final-system figures.

In [8]:
rows = []
for c, where, k, n, scope in (('AI caught, typos 10% (test)', '13 / README / MODEL_CARD', 468, 473, 'old single model at the 1% logit threshold (logit 4.67) only; NOT a statement about the final system'),
                             ('AI caught, T5 heavy paraphrase (test)', '13 / README / MODEL_CARD', 456, 473, 'old single model at the 1% logit threshold only; NOT a statement about the final system'),
                             ('humans flagged, after T5 light paraphrase', '13 / README / MODEL_CARD', 8, 300, 'old single model at the 1% logit threshold only')):
    p, lo, hi = wil(k, n); rows.append(dict(claim=c, where=where, k=k, n=n, value_pct=p, lo=lo, hi=hi, status='scope-limited: old system', note=scope))
for _, r in RES.iterrows():
    if r.attack == 'original': continue
    ai = r.group.startswith('AI')
    rows.append(dict(claim=f"{'AI flagged' if ai else 'students flagged'} by the official ensemble, {r.attack} ({r.group})", where='22b', k=r.flagged_k, n=r.n, value_pct=r.flagged_pct, lo=r.flagged_lo, hi=r.flagged_hi,
                     status='final system, descriptive' if 'held-out' in r.group else 'final system', note='random sample; on-topic essays'))
    rows.append(dict(claim=f"verdict flip, {r.attack} ({r.group})", where='22b', k=r.flip_k, n=r.flip_denominator, value_pct=r.flip_pct, lo=r.flip_lo, hi=r.flip_hi,
                     status='final system, descriptive' if 'held-out' in r.group else 'final system',
                     note='AI: flagged before, not flagged after' if ai else 'students: not flagged before, flagged after'))
CA = pd.DataFrame(rows); CA.to_csv(R + '22b_claims_audit.csv', index=False); print(len(CA), 'audit rows')

51 audit rows


## Key takeaways

1. **Notebook 13's robustness figures apply to the old single model at its old 1% logit threshold only.** The final system's strict flag line is much less robust. Official ensemble, *flagged*, random on-topic samples (old single model in brackets): test AI 99.2% unedited, 96.6% with 5% typos (99.4), 90.1% with 10% typos (98.9), 98.9% with light T5 (99.6), 81.4% with heavy T5 (96.4), 87.7% with T5 + 5% typos (98.9). Unseen-generator AI (descriptive): 94.7% unedited, 83.7% / 67.3% / 43.7% with 2 / 5 / 10% typos (old: 100 / 99.3 / 98.3), 93.3% light T5, 52.7% heavy T5.
2. **Most of what is lost moves to "above usual", not to "not flagged".** Flagged-or-above-usual stays at 99.4% (5% typos) and 98.7% (10%) for test AI and 96.0% under heavy T5; for unseen generators 92.7%, 86.0% and 84.7%. This is the case for showing the zone and not a yes/no.
3. **Verdict flips (flagged before, not flagged after), paired.** Test AI: 5% typos 2.6% [1.5, 4.4] (12 of 469); 10% typos 9.2% [6.9, 12.1]; light T5 0.2%; heavy T5 17.9% [14.7, 21.6]. Unseen generators: 5% typos 28.9% [23.9, 34.4] (82 of 284); 10% typos 53.9% [48.1, 59.6]; heavy T5 44.4%.
4. **The jury moves with the verdict.** Mean number of the 20 jurors flagging a flagged test AI essay falls from 19.8 to 18.0 (5% typos), 16.0 (10%), 16.5 (heavy T5); unseen generators 18.6 to 12.2, 8.8 and 12.3.
5. **Students are not harmed by the attacks.** No test student (0 of 300) is flagged under any attack; on held-out prompts 1 of 300 is flagged unedited (0.3%) and 3 (1.0%) after light T5. Light T5 does push some students to "above usual": 0.0% to 3.0% on test, 4.7% to 7.7% on held-out prompts. Typos lower the above-usual share for students (4.7% to 0.7% at 5% typos on held-out prompts), consistent with typos being a student signal.
6. **Phase 2's 13.5% was a mixture** (see `22_jury.ipynb` 10c): the flag is lost mostly on unseen prompts and unseen generators, not on familiar-topic AI.

**Proposed wording, NOT applied (README row and MODEL_CARD "Rewriting" bullet):**
> *Typos and rewriting (final system, [22b](Notebooks/22b_attack_final.ipynb)).* On random on-topic samples the official verdict flags 99.2% of AI essays on familiar topics when unedited, 96.6% with typos in 5% of words, 90.1% with 10% and 81.4% after heavy T5 paraphrasing (43% of words changed). For AI models the detectors never saw it is 94.7% unedited, 67.3% with 5% typos and 43.7% with 10%. Most lost flags become "above usual": flagged or above usual stays at 99.4% / 98.7% / 96.0% on familiar topics and 92.7% / 86.0% / 84.7% on unseen generators. The earlier figure of 98.9% under 10% typos belongs to the older single model at its old threshold. No student essay in a 300-essay test sample was flagged under any attack.

**Page copy for "Re-run the trial" (values read from `results/22b_attack_final.csv`, never typed):** "Light typos (5% of words) removed the flag from {flip_pct, AI test, typos 5%}% of flagged AI essays in our tests ({flip_k} of {flip_denominator}), and from {flip_pct, AI held-out generators, typos 5%}% of essays from AI models the detectors had not seen. Most of those landed at 'above usual', not 'not flagged'."

## Exploratory (not the official system): abstention "flag only if the official verdict flags AND at least k of 20 jurors flag"

- **The declared rule chose k = 14** (smallest k for which at least half the validation students flagged are demoted and validation AI caught falls by at most 1 point; it is also met for every larger k). It rests on **4 validation students**: k = 13 demotes 1 of them, k = 14 demotes 3, so the choice is fragile.
- **Test:** students flagged 2 of 2,291 before and after (0 demoted); AI caught 1,201 to 1,200 of 1,210 (1 demoted).
- **Held-out prompts (descriptive):** students flagged 31 to 17 of 4,464 (0.69% to 0.38%), 14 demoted to above usual (above-usual students 116 to 130); AI caught 2,660 to 2,658 of 2,671.
- **Held-out generators (descriptive):** AI caught 95.66% to 94.16%, 41 demoted to above usual (94 to 135).
- Reading: on unseen prompts the rule halves the students' false flags at a small cost in AI caught, but it is a post-hoc rule tuned on four essays and is labelled exploratory everywhere.
- **Status (decision 2026-11-09):** abstention stays exploratory and off the page. The choice of k rests on 4 validation students; confirming it would need data not yet used (a new set of essays). No claim about it should be made until then.

## Exploratory: an abstention rule — flag only if the official verdict flags AND at least k of 20 jurors flag

**Exploratory. Not the official system. Declared in `docs/jury_22b_declaration.md` before any number below.**
Idea: when the official verdict flags an essay but most of the jury does not, show *above usual* instead of *flagged*. k is chosen on validation by a rule fixed in advance: the smallest k such that (a) at least half of the validation students that the official verdict flags are demoted and (b) validation AI caught falls by at most 1 point. If no k meets both, the whole curve is reported and no k is chosen.
Caveat from the declaration: validation is also what every juror was calibrated on, and only about 5 validation students are flagged.

In [9]:
pe = pd.read_csv(R + '22_jury_per_essay.csv')
def curve(df, k):
    f = df.ensemble_flagged.to_numpy(); keep = f & (df.n_flag_20.to_numpy() >= k)
    return f, keep
val = pe[pe.split == 'val']; vs = val[val.label == 0]; va = val[val.label == 1]
rows = []
for k in range(1, 21):
    fs, ks = curve(vs, k); fa, ka = curve(va, k)
    rows.append(dict(k=k, val_students_flagged_official=int(fs.sum()), val_students_demoted=int((fs & ~ks).sum()), val_AI_caught_official_pct=100 * fa.mean(),
                     val_AI_caught_with_rule_pct=100 * ka.mean(), val_AI_demoted=int((fa & ~ka).sum())))
CV = pd.DataFrame(rows); CV['ok_a_half_students_demoted'] = CV.val_students_demoted * 2 >= CV.val_students_flagged_official
CV['ok_b_AI_loss_le_1pt'] = (CV.val_AI_caught_official_pct - CV.val_AI_caught_with_rule_pct) <= 1.0
CV['rule_met'] = CV.ok_a_half_students_demoted & CV.ok_b_AI_loss_le_1pt
CV.round(3).to_csv(R + '22b_abstention_val_curve.csv', index=False)
print(CV.round(2).to_string(index=False))
K = int(CV.loc[CV.rule_met, 'k'].min()) if CV.rule_met.any() else None
print('\nchosen k:', K)

 k  val_students_flagged_official  val_students_demoted  val_AI_caught_official_pct  val_AI_caught_with_rule_pct  val_AI_demoted  ok_a_half_students_demoted  ok_b_AI_loss_le_1pt  rule_met
 1                              4                     0                       99.01                        99.01               0                       False                 True     False
 2                              4                     0                       99.01                        99.01               0                       False                 True     False
 3                              4                     0                       99.01                        99.01               0                       False                 True     False
 4                              4                     0                       99.01                        99.01               0                       False                 True     False
 5                              4                     0     

In [10]:
rows = []
if K is not None:
    for name, mk in (('test students', (pe.split == 'test') & (pe.label == 0)), ('test AI', (pe.split == 'test') & (pe.label == 1)),
                     ('held-out prompt students [descriptive]', (pe.split == 'heldout_prompt') & (pe.label == 0)), ('held-out prompt AI [descriptive]', (pe.split == 'heldout_prompt') & (pe.label == 1)),
                     ('held-out generator AI [descriptive]', pe.split == 'heldout_generator')):
        d = pe[mk]; n = len(d); f = d.ensemble_flagged.to_numpy(); keep = f & (d.n_flag_20.to_numpy() >= K)
        for kk in sorted({1, K}):
            kp = f & (d.n_flag_20.to_numpy() >= kk)
            a = wil(f.sum(), n); b = wil(kp.sum(), n)
            rows.append(dict(population=name, n=n, k=kk, official_flagged=int(f.sum()), official_pct=a[0], rule_flagged=int(kp.sum()), rule_pct=b[0], rule_lo=b[1], rule_hi=b[2],
                             demoted_to_above_usual=int((f & ~kp).sum()), above_usual_official=int((d.ensemble_p.between(TAU_FLAG, TAU_ABOVE, inclusive='right')).sum())))
    AB = pd.DataFrame(rows); AB['above_usual_after_rule'] = AB.above_usual_official + AB.demoted_to_above_usual
    AB.round(3).to_csv(R + '22b_abstention_report.csv', index=False); print(AB[AB.k == K].round(2).to_string(index=False))
else:
    print('no k satisfies the declared rule; the validation curve above is the result and no k is applied to test or held-out data')

                            population    n  k  official_flagged  official_pct  rule_flagged  rule_pct  rule_lo  rule_hi  demoted_to_above_usual  above_usual_official  above_usual_after_rule
                         test students 2291 14                 2          0.09             2      0.09     0.02     0.32                       0                    16                      16
                               test AI 1210 14              1201         99.26          1200     99.17    98.49    99.55                       1                     3                       4
held-out prompt students [descriptive] 4464 14                31          0.69            17      0.38     0.24     0.61                      14                   116                     130
      held-out prompt AI [descriptive] 2671 14              2660         99.59          2658     99.51    99.17    99.72                       2                     9                      11
   held-out generator AI [descriptive] 2721 1